# ITAP ML Pipeline v3: Hybrid Threat Prediction, Anomaly Detection & Malware Classification
This notebook trains **three** deep learning models for the ITAP Security Platform:
1. **LSTM Exploit Predictor**: Predicts the likelihood of a CVE being exploited. Trained on CISA KEV + 50K synthetic CVSS profiles.
2. **Autoencoder Anomaly Detector**: Identifies zero-day network patterns. Trained on KDD Cup '99 + 100K synthetic flows.
3. **Malware Classifier (NEW)**: Identifies malware families from byte patterns. Trained on the Microsoft BIG2015 dataset.

**All models use `ModelCheckpoint` callbacks so training resumes automatically if interrupted.**

*Hardware: Kaggle Cloud GPU (T4 x2)*

## Section 1: Environment Setup & Dependencies

In [ ]:
!pip install tensorflow scikit-learn requests pandas numpy tqdm pefile imbalanced-learn

In [ ]:
import os
import time
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model, load_model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, LSTM

CKPT_DIR = '/kaggle/working/checkpoints'
WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('Directories created successfully.')


## Section 2: LSTM Exploit Predictor — Data (CISA KEV + Synthetic CVSS)

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import os

print("1. Loading UNSW-NB15 Dataset from Kaggle...")
train_path = "/kaggle/input/unsw-nb15/UNSW_NB15_training-set.csv"
test_path = "/kaggle/input/unsw-nb15/UNSW_NB15_testing-set.csv"

# If Kaggle structure differs, search for it
if not os.path.exists(train_path):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "UNSW_NB15_training-set.csv" in files:
            train_path = os.path.join(root, "UNSW_NB15_training-set.csv")
        if "UNSW_NB15_testing-set.csv" in files:
            test_path = os.path.join(root, "UNSW_NB15_testing-set.csv")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_unsw = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# Fill NaNs in attack_cat with 'Normal'
df_unsw['attack_cat'] = df_unsw['attack_cat'].fillna('Normal')
df_unsw.loc[df_unsw['attack_cat'] == 'Normal', 'attack_cat'] = 'Normal'

print(f"Loaded {len(df_unsw)} total network flows.")

# Drop id and label
df_unsw = df_unsw.drop(['id'], axis=1)

# Separate Target
y_attack = df_unsw['attack_cat'].values
df_unsw = df_unsw.drop(['attack_cat', 'label'], axis=1)

# Encode categorical
categorical_cols = ['proto', 'service', 'state']
df_unsw = pd.get_dummies(df_unsw, columns=categorical_cols)

# Scale
scaler = MinMaxScaler()
X_unsw_scaled = scaler.fit_transform(df_unsw)

# Label Encode Target
le = LabelEncoder()
y_unsw_encoded = le.fit_transform(y_attack)
num_attack_classes = len(le.classes_)
print(f"Attack Categories ({num_attack_classes}):", list(le.classes_))

# LSTM requires shape (samples, timesteps, features)
X_lstm = X_unsw_scaled.reshape((X_unsw_scaled.shape[0], 1, X_unsw_scaled.shape[1]))

X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_unsw_encoded, test_size=0.2, random_state=42, stratify=y_unsw_encoded)
print(f"\nLSTM Training set: {X_train.shape}")


## Section 3: Train LSTM Exploit Predictor (with Checkpointing)

In [ ]:

LSTM_CKPT_PATH  = f"{CKPT_DIR}/lstm_best.weights.h5"
LSTM_FINAL_PATH = f"{WEIGHTS_DIR}/itap_lstm_v2.h5"

def build_lstm_model(input_shape, num_classes):
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
    from tensorflow.keras.models import Sequential
    
    model = Sequential([
        LSTM(128, activation='relu', input_shape=input_shape, return_sequences=True),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Resume from checkpoint if it exists
if os.path.exists(LSTM_FINAL_PATH):
    print("[RESUME] Loading existing LSTM model from final weights...")
    from tensorflow.keras.models import load_model
    lstm_model = load_model(LSTM_FINAL_PATH)
else:
    print("Building fresh LSTM Network Attack Classifier...")
    lstm_model = build_lstm_model((1, X_train.shape[2]), num_attack_classes)

if os.path.exists(LSTM_CKPT_PATH):
    print(f"[RESUME] Loading LSTM checkpoint weights from {LSTM_CKPT_PATH}")
    lstm_model.load_weights(LSTM_CKPT_PATH)

lstm_model.summary()

lstm_callbacks = [
    ModelCheckpoint(filepath=LSTM_CKPT_PATH, monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

# Using class weights instead of SMOTE!
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

print("\nStarting UNSW-NB15 LSTM Training (GPU accelerated)...")
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=256,
    validation_data=(X_test, y_test),
    callbacks=lstm_callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Final evaluation
loss, acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"\nLSTM Final Test Accuracy: {acc*100:.2f}%")


## Section 4: Autoencoder Anomaly Detector — Data (KDD Cup '99 + Synthetic)

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import gc

print("--- REUSING UNSW-NB15 DATASET FOR AUTOENCODER ---")
print("Cleaning up LSTM memory but keeping scaled data...")

# Free memory from LSTM structures
for var in ['X_train', 'y_train', 'X_lstm']:
    if var in globals():
        del globals()[var]
gc.collect()

print("Extracting 'Normal' traffic from UNSW-NB15 for Anomaly Detection...")
# y_attack and X_unsw_scaled are still in memory from the LSTM data cell!
normal_indices = (y_attack == 'Normal')
X_normal = X_unsw_scaled[normal_indices]

# Now we can safely delete X_unsw_scaled and y_attack
del X_unsw_scaled, y_attack, normal_indices
gc.collect()

X_net_train, X_net_test = train_test_split(X_normal, test_size=0.2, random_state=123)
del X_normal
gc.collect()

print(f"\nAutoencoder Training set (UNSW Normal traffic only): {X_net_train.shape}")


## Section 5: Train Autoencoder Anomaly Detector (with Checkpointing)

In [ ]:

AE_CKPT_PATH  = f"{CKPT_DIR}/ae_best.weights.h5"
AE_FINAL_PATH = f"{WEIGHTS_DIR}/itap_autoencoder_v2.h5"

def build_autoencoder(input_dim):
    from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
    from tensorflow.keras.models import Model
    
    input_layer  = Input(shape=(input_dim,))
    encoded      = Dense(128, activation='relu')(input_layer)
    encoded      = BatchNormalization()(encoded)
    encoded      = Dense(64, activation='relu')(encoded)
    encoded      = Dense(32, activation='relu')(encoded)  # Bottleneck
    
    decoded      = Dense(64, activation='relu')(encoded)
    decoded      = BatchNormalization()(decoded)
    decoded      = Dense(128, activation='relu')(decoded)
    output_layer = Dense(input_dim, activation='sigmoid')(decoded) # Assuming MinMax scaled 0-1
    
    autoencoder  = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

if os.path.exists(AE_FINAL_PATH):
    print("[RESUME] Loading existing Autoencoder from final weights...")
    from tensorflow.keras.models import load_model
    autoencoder = load_model(AE_FINAL_PATH)
else:
    print("Building fresh Autoencoder...")
    autoencoder = build_autoencoder(X_net_train.shape[1])

if os.path.exists(AE_CKPT_PATH):
    print(f"[RESUME] Loading Autoencoder checkpoint weights from {AE_CKPT_PATH}")
    autoencoder.load_weights(AE_CKPT_PATH)

autoencoder.summary()

ae_callbacks = [
    ModelCheckpoint(filepath=AE_CKPT_PATH, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

print("\nStarting UNSW-NB15 Autoencoder Training...")
ae_history = autoencoder.fit(
    X_net_train, X_net_train,
    epochs=100,
    batch_size=256,
    validation_data=(X_net_test, X_net_test),
    callbacks=ae_callbacks,
    verbose=1
)

val_loss = autoencoder.evaluate(X_net_test, X_net_test, verbose=0)
print(f"\nAutoencoder Final Validation MSE Loss: {val_loss:.6f}")


## Section 6: NEW — Malware Classifier Data (Microsoft BIG2015 Dataset)
This dataset contains **10,868 malware samples** from 9 families as raw byte sequences and disassembly dumps.
We train a **1D CNN** on the byte histograms (256 features) to classify the malware family with high accuracy.

In [ ]:
import glob

# Extremely robust dataset discovery
KAGGLE_INPUT = "/kaggle/input"
MALWARE_DATASET_PATH = None
TRAIN_LABELS_PATH = None
SKIP_MALWARE = False

print("Scanning all directories in /kaggle/input for malware dataset...")
# We will search for trainLabels.csv and .bytes files no matter how deep they are nested
for root, dirs, files in os.walk(KAGGLE_INPUT):
    for fname in files:
        if "trainlabel" in fname.lower() or fname.lower() == "trainlabels.csv":
            TRAIN_LABELS_PATH = os.path.join(root, fname)
            MALWARE_DATASET_PATH = root  # The folder containing labels
            break
    if TRAIN_LABELS_PATH:
        break

if MALWARE_DATASET_PATH is None:
    print("WARNING: Malware dataset (trainLabels.csv) not found anywhere in /kaggle/input! Skipping malware training.")
    SKIP_MALWARE = True
else:
    print(f"Found Malware Dataset Folder: {MALWARE_DATASET_PATH}")
    print(f"Found Labels CSV: {TRAIN_LABELS_PATH}")

MALWARE_FAMILIES = [
    "Ramnit", "Lollipop", "Kelihos_v3", "Vundo", "Simda",
    "Tracur", "Kelihos_v1", "Obfuscator.ACY", "Gatak"
]


In [ ]:
def load_malware_data(labels_path, max_samples=5000):
    print("Loading Microsoft BIG2015 Malware Dataset...")
    labels_df = pd.read_csv(labels_path)
    print("   -> Found", len(labels_df), "total labeled malware samples.")
    print("   -> Class distribution:")
    print(labels_df['Class'].value_counts())

    # Deep scan for .bytes files
    bytes_dir = None
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        bfiles = [fname for fname in files if fname.endswith(".bytes")]
        if len(bfiles) > 10:
            bytes_dir = root
            print("   -> Found .bytes files in:", bytes_dir, "total:", len(bfiles))
            break
            
    if bytes_dir is None:
        raise FileNotFoundError("Could not find .bytes files anywhere in /kaggle/input!")

    if len(labels_df) > max_samples:
        labels_df = labels_df.groupby("Class", group_keys=False).apply(
            lambda x: x.sample(min(len(x), max_samples // 9), random_state=42)
        ).reset_index(drop=True)
        print("   -> Sampled down to", len(labels_df), "samples (balanced).")

    X, y = [], []
    for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Reading byte files"):
        sample_id = row["Id"]
        label = int(row["Class"]) - 1
        bytes_file = os.path.join(bytes_dir, str(sample_id) + ".bytes")
        if not os.path.exists(bytes_file):
            continue
        try:
            with open(bytes_file, "r", errors="ignore") as bf:
                content = bf.read()
            hex_values = [
                int(b, 16) for b in content.split()
                if len(b) == 2 and all(c in "0123456789abcdefABCDEF" for c in b) and b != "??"
            ]
            if len(hex_values) < 100:
                continue
            histogram, _ = np.histogram(hex_values[:50000], bins=256, range=(0, 255))
            histogram = histogram.astype(float) / (histogram.sum() + 1e-8)
            X.append(histogram)
            y.append(label)
        except Exception:
            continue

    X = np.array(X)
    y = np.array(y)
    print("Malware dataset loaded:", X.shape[0], "samples,", X.shape[1], "features each.")
    return X, y

if not SKIP_MALWARE:
    try:
        X_mal, y_mal = load_malware_data(TRAIN_LABELS_PATH, max_samples=5000)
        X_mal_train, X_mal_test, y_mal_train, y_mal_test = train_test_split(
            X_mal, y_mal, test_size=0.2, random_state=42, stratify=y_mal
        )
        print("Malware Training set shape:", X_mal_train.shape)
        unique_classes = sorted(set(int(v) for v in y_mal.tolist()))
        print("Families found:", [MALWARE_FAMILIES[i] for i in unique_classes])
    except Exception as ex:
        print("ERROR loading malware data:", ex)
        SKIP_MALWARE = True
else:
    print("Malware dataset skipped - dataset not available.")


In [ ]:
if not SKIP_MALWARE:
    csv_path = None
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for f in files:
            if f == "LargeTrain.csv":
                csv_path = os.path.join(root, f)
                break
                
    if csv_path:
        try:
            print(f"Loading preprocessed features from {csv_path}...")
            df = pd.read_csv(csv_path)
            print(f"Dataset Shape: {df.shape}")
            
            # Select only numeric columns just in case
            df = df.select_dtypes(include=[np.number])
            
            y = df['Class'].values - 1  # Kaggle labels are 1-9 -> Keras 0-8
            X = df.drop('Class', axis=1).values
            
            X_mal_train, X_mal_val, y_mal_train, y_mal_val = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            print(f"Split: {X_mal_train.shape[0]} Train | {X_mal_val.shape[0]} Val")
            
            num_features = X_mal_train.shape[1]
            unique_classes = np.unique(y).tolist()
            print("Families found:", [MALWARE_FAMILIES[i] for i in unique_classes])
        except Exception as ex:
            print("ERROR loading tabular malware data:", ex)
            SKIP_MALWARE = True
    else:
        print("Malware dataset skipped - LargeTrain.csv not found.")
        SKIP_MALWARE = True
else:
    print("Malware dataset skipped.")


In [ ]:
if not SKIP_MALWARE:
    MAL_CKPT_PATH  = CKPT_DIR + "/malware_best.weights.h5"
    MAL_FINAL_PATH = WEIGHTS_DIR + "/itap_malware_classifier_v1.h5"
    NUM_CLASSES = len(unique_classes)

    def build_malware_mlp(input_dim, num_classes):
        input_layer = Input(shape=(input_dim,))
        x = Dense(512, activation="relu")(input_layer)
        x = BatchNormalization()(x)
        x = Dropout(0.4)(x)
        x = Dense(256, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(0.3)(x)
        x = Dense(128, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)
        output = Dense(num_classes, activation="softmax")(x)
        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    if os.path.exists(MAL_FINAL_PATH):
        print("[RESUME] Loading existing Malware Classifier...")
        malware_model = load_model(MAL_FINAL_PATH)
    else:
        print("Building fresh Tabular Malware MLP...")
        malware_model = build_malware_mlp(num_features, NUM_CLASSES)

    if os.path.exists(MAL_CKPT_PATH):
        print("[RESUME] Loading Malware checkpoint weights from", MAL_CKPT_PATH)
        malware_model.load_weights(MAL_CKPT_PATH)

    malware_model.summary()

    mal_callbacks = [
        ModelCheckpoint(filepath=MAL_CKPT_PATH, monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
    ]

    print("Starting Microsoft BIG2015 Tabular Malware Training (MLP on GPU)...")
    malware_history = malware_model.fit(
        X_mal_train, y_mal_train,
        epochs=150, 
        batch_size=128,
        validation_data=(X_mal_val, y_mal_val),
        callbacks=mal_callbacks,
        verbose=1
    )

    loss, acc = malware_model.evaluate(X_mal_val, y_mal_val, verbose=0)
    print("Malware MLP Final Test Accuracy:", round(acc*100, 2), "%")
    
    final_malware_acc = float(acc)
else:
    print("Malware training skipped.")
    malware_model = None
    MAL_FINAL_PATH = None
    final_malware_acc = None


if not SKIP_MALWARE:
    MAL_CKPT_PATH  = CKPT_DIR + "/malware_best.weights.h5"
    MAL_FINAL_PATH = WEIGHTS_DIR + "/itap_malware_classifier_v1.h5"
    NUM_CLASSES = len(unique_classes)

    def build_malware_cnn(input_dim, num_classes):
        input_layer = Input(shape=(input_dim, 1))
        x = Conv1D(64,  kernel_size=8, activation="relu", padding="same")(input_layer)
        x = Conv1D(128, kernel_size=4, activation="relu", padding="same")(x)
        x = Conv1D(256, kernel_size=2, activation="relu", padding="same")(x)
        x = GlobalMaxPooling1D()(x)
        x = Dense(256, activation="relu")(x)
        x = Dropout(0.4)(x)
        x = Dense(128, activation="relu")(x)
        x = Dropout(0.3)(x)
        output = Dense(num_classes, activation="softmax")(x)
        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return model

    if os.path.exists(MAL_FINAL_PATH):
        print("[RESUME] Loading existing Malware Classifier from final weights...")
        malware_model = load_model(MAL_FINAL_PATH)
    else:
        print("Building fresh Malware Classifier CNN...")
        malware_model = build_malware_cnn(256, NUM_CLASSES)

    if os.path.exists(MAL_CKPT_PATH):
        print("[RESUME] Loading Malware checkpoint weights from", MAL_CKPT_PATH)
        malware_model.load_weights(MAL_CKPT_PATH)

    malware_model.summary()

    mal_callbacks = [
        ModelCheckpoint(filepath=MAL_CKPT_PATH, monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
    ]

    print("Starting Microsoft BIG2015 Malware Classification Training (FULL DATASET ON GPU)...")
    malware_history = malware_model.fit(
        train_gen,
        epochs=15,          # Reduced epochs because full dataset is massive
        validation_data=val_gen,
        callbacks=mal_callbacks,
        verbose=1
    )

    loss, acc = malware_model.evaluate(val_gen, verbose=0)
    print("Malware Classifier Final Test Accuracy:", round(acc*100, 2), "%")
    
    # Store val acc for metadata
    final_malware_acc = float(acc)
else:
    print("Malware CNN training skipped - dataset not available.")
    malware_model = None
    MAL_FINAL_PATH = None
    final_malware_acc = None


print("Saving Hybrid LSTM Exploit Predictor...")
lstm_model.save(LSTM_FINAL_PATH)
print("  -> Saved to", LSTM_FINAL_PATH)

print("Saving Hybrid Autoencoder Anomaly Detector...")
autoencoder.save(AE_FINAL_PATH)
print("  -> Saved to", AE_FINAL_PATH)

metadata = {
    'lstm_accuracy': float(lstm_model.evaluate(X_test, y_test, verbose=0)[1]),
    'autoencoder_val_loss': float(autoencoder.evaluate(X_net_test, X_net_test, verbose=0)),
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

if not SKIP_MALWARE and malware_model is not None:
    print("Saving Microsoft BIG2015 Malware Classifier...")
    malware_model.save(MAL_FINAL_PATH)
    print("  -> Saved to", MAL_FINAL_PATH)
    metadata['malware_accuracy'] = final_malware_acc
    metadata['malware_families'] = MALWARE_FAMILIES
else:
    print("Malware model not saved (was skipped).")
    metadata['malware_accuracy'] = None

with open(WEIGHTS_DIR + '/model_metadata.json', 'w') as mf:
    json.dump(metadata, mf, indent=2)

print("=== ALL ITAP v3 MODELS TRAINED AND EXPORTED SUCCESSFULLY ===")
print("  LSTM Accuracy        :", round(metadata["lstm_accuracy"]*100, 2), "%")
print("  Autoencoder MSE Loss :", round(metadata["autoencoder_val_loss"], 6))
mal_acc = metadata.get('malware_accuracy')
print("  Malware Accuracy     :", str(round(mal_acc*100, 2)) + "%" if mal_acc else "Skipped")
print("Run: python ai_training/push_to_kaggle.py --download")


In [ ]:
print("Saving Hybrid LSTM Exploit Predictor...")
lstm_model.save(LSTM_FINAL_PATH)
print("  -> Saved to", LSTM_FINAL_PATH)

print("Saving Hybrid Autoencoder Anomaly Detector...")
autoencoder.save(AE_FINAL_PATH)
print("  -> Saved to", AE_FINAL_PATH)

metadata = {
    'lstm_accuracy': float(lstm_model.evaluate(X_test, y_test, verbose=0)[1]),
    'autoencoder_val_loss': float(autoencoder.evaluate(X_net_test, X_net_test, verbose=0)),
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

if not SKIP_MALWARE and malware_model is not None:
    print("Saving Microsoft BIG2015 Tabular Classifier...")
    malware_model.save(MAL_FINAL_PATH)
    print("  -> Saved to", MAL_FINAL_PATH)
    metadata['malware_accuracy'] = final_malware_acc
    metadata['malware_families'] = MALWARE_FAMILIES
else:
    print("Malware model not saved (was skipped).")
    metadata['malware_accuracy'] = None

with open(WEIGHTS_DIR + '/model_metadata.json', 'w') as mf:
    json.dump(metadata, mf, indent=2)

print("=== ALL ITAP v3 MODELS TRAINED AND EXPORTED SUCCESSFULLY ===")
print("  LSTM Accuracy        :", round(metadata["lstm_accuracy"]*100, 2), "%")
print("  Autoencoder MSE Loss :", round(metadata["autoencoder_val_loss"], 6))
mal_acc = metadata.get('malware_accuracy')
print("  Malware Accuracy     :", str(round(mal_acc*100, 2)) + "%" if mal_acc else "Skipped")
print("Run: python ai_training/push_to_kaggle.py --download")
